# Hopfield Associative Memory: Load, Lesions, and Retrieval Errors
This notebook explores the dynamics of a classical Hopfield network, demonstrates energy descent, synaptic lesions, and visualizes retrieval error distributions.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from src.patterns import generate_random_bipolar_patterns
from src.hopfield import HopfieldNetwork
from src.lesions import apply_random_lesion, corrupt_cue
from src.metrics import evaluate_trial
from src.plots import set_plot_style
set_plot_style()

## 1. Network Initialization & Weight Matrix
Train a network with =100$ and =5$ patterns.

In [ ]:
N, P = 100, 5
rng = np.random.default_rng(42)
patterns = generate_random_bipolar_patterns(N, P, rng=rng)
net = HopfieldNetwork(N)
net.train(patterns)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(net.W, cmap="coolwarm", aspect="auto")
ax.set_title("Hebbian Weight Matrix (Zero Diagonal)")
plt.colorbar(im, ax=ax)
plt.show()

## 2. Synaptic Lesioning
Delete 30% of undirected synaptic connections and inspect weight matrix.

In [ ]:
W_lesioned, deleted = apply_random_lesion(net.W, lesion_fraction=0.3, rng=rng)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.imshow(net.W, cmap="coolwarm")
ax1.set_title("Intact Network")
ax2.imshow(W_lesioned, cmap="coolwarm")
ax2.set_title(f"Lesioned Network (f=0.30, {deleted} edges deleted)")
plt.show()

## 3. Retrieval Dynamics & Monotonic Energy Descent

In [ ]:
cue = corrupt_cue(patterns[0], flip_probability=0.25, rng=rng)
final_state, converged, sweeps, energy_hist = net.retrieve(cue, track_energy=True, rng=rng)

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(energy_hist, marker=".", color="tab:blue")
ax.set_xlabel("Neuron Update Step")
ax.set_ylabel("Lyapunov Energy E(s)")
ax.set_title("Monotonic Energy Descent")
plt.show()
print(f"Target recall exact: {np.array_equal(final_state, patterns[0])}")